# Phase 5e: Complete Braided MTCs — Technical Notebook

**Phase:** 5e (Braided MTCs + Topological Invariants)  
**Date:** April 2026  
**Paper:** Paper 11 — *U_q(sl_2) in Lean 4*

First complete verified braided fusion categories in any proof assistant:
- **Ising**: 6 hexagon equations, 4 ribbon conditions, Gauss sum, trefoil = -1
- **Fibonacci**: 3 hexagon equations (E1-E3), twist, writhe removal
- **SU(2)_3**: S-matrix unitarity over novel quartic field Q[x]/(20x^4-10x^2+1)
- **SU(2)_4**: S-matrix unitarity over Q(sqrt(3))
- All verified by native_decide over custom number fields. Zero sorry.

---

In [1]:
import numpy as np
from src.core.formulas import (
    ising_r_matrix, ising_twist, ising_f_symbol,
    fibonacci_r_matrix, fibonacci_twist, trefoil_jones_ising,
    su2k_fusion_rule, su2k_quantum_dim, su2k_s_matrix_entry,
)
from src.core.visualizations import COLORS

## 1. Ising Braiding: R-matrix and Hexagon Equations

The Ising MTC has R-matrix eigenvalues in Q(zeta_16), the 16th cyclotomic field.

In [2]:
# R-matrix data
print("Ising R-matrix eigenvalues:")
for (a,b,c), label in [((1,1,0), 'R^{ss}_1'), ((1,1,2), 'R^{ss}_psi'),
                         ((1,2,1), 'R^{sp}_s'), ((2,2,0), 'R^{pp}_1')]:
    R = ising_r_matrix(a, b, c)
    print(f"  {label} = {R:.6f}  |R| = {abs(R):.6f}")

print("\nTwist factors:")
for a, label in [(0, '1'), (1, 'sigma'), (2, 'psi')]:
    print(f"  theta_{label} = {ising_twist(a):.6f}")

Ising R-matrix eigenvalues:
  R^{ss}_1 = 0.923880-0.382683j  |R| = 1.000000
  R^{ss}_psi = 0.382683+0.923880j  |R| = 1.000000
  R^{sp}_s = -0.000000-1.000000j  |R| = 1.000000
  R^{pp}_1 = -1.000000  |R| = 1.000000

Twist factors:
  theta_1 = 1.000000
  theta_sigma = 0.923880+0.382683j
  theta_psi = -1.000000


In [3]:
# Hexagon equations verification
R1 = ising_r_matrix(1, 1, 0)
Rpsi = ising_r_matrix(1, 1, 2)
Rsp = ising_r_matrix(1, 2, 1)
inv_sqrt2 = 1/np.sqrt(2)

print("Hexagon equations (Lean: hexagon_I/II/III in IsingBraiding.lean):")
print(f"  H-I:   (1/sqrt2)(1+Rsp) = R1^2:  {abs(inv_sqrt2*(1+Rsp) - R1**2):.2e}")
print(f"  H-II:  (1/sqrt2)(1-Rsp) = R1*Rpsi:  {abs(inv_sqrt2*(1-Rsp) - R1*Rpsi):.2e}")
print(f"  H-III: (1/sqrt2)(1+Rsp) = -Rpsi^2:  {abs(inv_sqrt2*(1+Rsp) + Rpsi**2):.2e}")

Hexagon equations (Lean: hexagon_I/II/III in IsingBraiding.lean):
  H-I:   (1/sqrt2)(1+Rsp) = R1^2:  1.11e-16
  H-II:  (1/sqrt2)(1-Rsp) = R1*Rpsi:  1.11e-16
  H-III: (1/sqrt2)(1+Rsp) = -Rpsi^2:  2.22e-16


## 2. Trefoil Knot Invariant

The Jones polynomial of the right-handed trefoil evaluated at q=i,
computed from the Ising R-matrix (no braid group needed for 2-strand braids).

In [4]:
result = trefoil_jones_ising()
print(f"RT(trefoil, sigma) = {result:.6f}")
print(f"Jones polynomial V(i) = -1")
print(f"Match: {abs(result - (-1)) < 1e-12}")
print()
print("Lean theorem: trefoil_eq_neg_sqrt2 (IsingBraiding.lean)")
print("Proof: native_decide over Q(zeta_16)")

RT(trefoil, sigma) = -1.000000+0.000000j
Jones polynomial V(i) = -1
Match: True

Lean theorem: trefoil_eq_neg_sqrt2 (IsingBraiding.lean)
Proof: native_decide over Q(zeta_16)


## 3. Fibonacci Braiding

R-matrix eigenvalues in Q(zeta_5). Hexagon equations E1-E3 all verified.

In [5]:
R1_fib = fibonacci_r_matrix(0)
Rtau = fibonacci_r_matrix(1)
theta_tau = fibonacci_twist()
phi = (1 + np.sqrt(5)) / 2
phi_inv = phi - 1

print("Fibonacci R-matrix:")
print(f"  R1 = {R1_fib:.6f} = e^{{-4pi*i/5}}")
print(f"  Rtau = {Rtau:.6f} = e^{{3pi*i/5}}")
print(f"  theta_tau = {theta_tau:.6f} = e^{{4pi*i/5}}")
print()
print("Hexagon equations (Lean: hexagon_E1/E2/E3 in QCyc5.lean):")
print(f"  E1: R1^2 = phi^-1 + Rtau:  {abs(R1_fib**2 - (phi_inv + Rtau)):.2e}")
print(f"  E3: Rtau^2 + phi^-1*Rtau + 1 = 0:  {abs(Rtau**2 + phi_inv*Rtau + 1):.2e}")
print(f"  R1 = Rtau^2:  {abs(R1_fib - Rtau**2):.2e}")

Fibonacci R-matrix:
  R1 = -0.809017-0.587785j = e^{-4pi*i/5}
  Rtau = -0.309017+0.951057j = e^{3pi*i/5}
  theta_tau = -0.809017+0.587785j = e^{4pi*i/5}

Hexagon equations (Lean: hexagon_E1/E2/E3 in QCyc5.lean):
  E1: R1^2 = phi^-1 + Rtau:  4.44e-16
  E3: Rtau^2 + phi^-1*Rtau + 1 = 0:  2.22e-16
  R1 = Rtau^2:  3.14e-16


## 4. SU(2)_3 and SU(2)_4 S-matrix Unitarity

S-matrix unitarity verified over:
- k=3: Q[x]/(20x^4-10x^2+1) — novel quartic field, conductor 40
- k=4: Q(sqrt(3)) — standard quadratic extension

In [6]:
for k in [3, 4]:
    n = k + 1
    S = np.array([[su2k_s_matrix_entry(k, i, j) for j in range(n)] for i in range(n)])
    I_check = S @ S.T
    max_dev = np.max(np.abs(I_check - np.eye(n)))
    det = np.linalg.det(S)
    print(f"SU(2)_{k}: {n}x{n} S-matrix")
    print(f"  S*S^T = I max deviation: {max_dev:.2e}")
    print(f"  det(S) = {det:.4f}")
    print(f"  Quantum dims: {[round(su2k_quantum_dim(k, j), 4) for j in range(n)]}")
    print()

SU(2)_3: 4x4 S-matrix
  S*S^T = I max deviation: 1.48e-16
  det(S) = 1.0000
  Quantum dims: [1.0, 1.618, 1.618, 1.0]

SU(2)_4: 5x5 S-matrix
  S*S^T = I max deviation: 4.72e-16
  det(S) = 1.0000
  Quantum dims: [1.0, 1.7321, 2.0, 1.7321, 1.0]



## 5. Provenance

| Lean Module | Theorems | Sorry | Key Result |
|-------------|----------|-------|------------|
| QCyc16.lean | 6 | 0 | Q(zeta_16) cyclotomic field |
| QCyc5.lean | 9 | 0 | Fibonacci hexagon E1-E3 |
| IsingBraiding.lean | 23 | 0 | 6 hexagons + 4 ribbons + trefoil |
| QSqrt3.lean | 8 | 0 | SU(2)_4 S-matrix unitarity |
| QLevel3.lean | 19 | 0 | SU(2)_3 S-matrix unitarity + d=phi |

All 65 theorems proved by `native_decide`. Zero sorry. Zero axioms.